In [72]:
!pip install -q gradio groq langchain langchain-community langchain-groq langchain-huggingface langchain-text-splitters faiss-cpu sentence-transformers pypdf


In [73]:
import os
from google.colab import userdata
os.environ['groq']=userdata.get('groq')
print("key is set")

key is set


In [75]:
from groq import Groq
client=Groq(api_key=os.environ['groq'])
response=client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {"role": "system", "content": "You are a expert football pundit"},
        {"role": "user", "content": "give me predictions and reasons for the champions for fifa"},
    ]
)
print(response.choices[0].message.content)

**2026 FIFA World Cup – Who’s Likely to Lift the Trophy and Why?**  
*(The tournament will be staged across the United States, Canada and Mexico in June‑July 2026. The analysis below is a forecast based on current form, player pools, coaching trends and the qualifying campaigns that are already underway. As always, football is notoriously unpredictable – injuries, form swings and the “magic of the World Cup” can overturn any projection.)*  

---

## 1. The Front‑Runners (≥ 15 % implied probability)

| Team | Why they’re a favourite | Key ingredients |
|------|------------------------|-----------------|
| **Brazil** | • Deep talent pool – every top‑four European league has at least two Brazilian starters. <br>• A blend of experienced winners (Neymar, Alisson, Casemiro) and a new generation of 21‑year‑olds (Endrick, Gabriel Martinelli, Rodrygo). <br>• Tactical flexibility under coach **Dorival Júnior** – comfortable in a 4‑2‑3‑1 or a high‑pressing 3‑5‑2. | • Attack: 3‑4 world‑class forwa

In [76]:
import gradio as gr
initial_system_prompt = '''You have indepth knowledge about formula 1 and everything related to it like cars,drivers,teams,logistics,budgets etc.
The level of your knowledge and game sense transcends team principals,drivers,pundits '''
def respond(message, history, system_prompt_input, temperature_input):
  messages = []
  if system_prompt_input:
      messages.append({"role": "system", "content": system_prompt_input})

  for chat_pair in history:
    if not isinstance(chat_pair, (list, tuple)):
        continue
    human_message = chat_pair[0] if len(chat_pair) > 0 else ""
    ai_message = chat_pair[1] if len(chat_pair) > 1 else ""

    if human_message:
        messages.append({"role": "user", "content": human_message})
    if ai_message is not None and ai_message != "":
        messages.append({"role": "assistant", "content": ai_message})

  messages.append({"role": "user", "content": message})

  response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=messages,
    temperature=temperature_input,
    stream=True
  )

  partial = ""
  for chunk in response:
    if chunk.choices[0].delta.content is not None:
      partial += chunk.choices[0].delta.content
      yield partial


system_prompt_textbox = gr.Textbox(label="System Prompt", value=initial_system_prompt, lines=5)
temperature_slider = gr.Slider(minimum=0.0, maximum=1.0, value=0.6, step=0.1, label="Temperature")

demo = gr.ChatInterface(
    fn=respond,
    type="messages",
    textbox=gr.Textbox(placeholder="Ask me anything about F1!", container=False, scale=7),
    title="F1 Chatbot",
    cache_examples=False,
    additional_inputs=[system_prompt_textbox, temperature_slider]
)

demo.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://951d211dbbe16088b8.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://951d211dbbe16088b8.gradio.live
